# Matrix Factorization Recommender

This notebook builds a lightweight movie recommendation engine using SVD-based collaborative filtering with the Surprise library.

The project uses the compact MovieLens dataset and is designed to run on a normal CPU without a GPU.

## 1. Imports and paths

In [1]:
from pathlib import Path
import zipfile
import joblib
import pandas as pd
import requests

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse, mae

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data'
MODEL_DIR = ROOT / 'models'
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

ZIP_URL = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
ZIP_PATH = DATA_DIR / 'ml-latest-small.zip'
EXTRACT_DIR = DATA_DIR / 'ml-latest-small'

print('Project root:', ROOT)

Project root: C:\Users\user\Desktop\Matrix-Factorization-Recommender


## 2. Download the small MovieLens dataset

The archive is only downloaded when it is missing. This keeps repeated notebook runs lightweight.

In [2]:
if not EXTRACT_DIR.exists():
    if not ZIP_PATH.exists():
        response = requests.get(ZIP_URL, timeout=60)
        response.raise_for_status()
        ZIP_PATH.write_bytes(response.content)

    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(DATA_DIR)

ratings_path = EXTRACT_DIR / 'ratings.csv'
movies_path = EXTRACT_DIR / 'movies.csv'

ratings = pd.read_csv(ratings_path)
movies = pd.read_csv(movies_path)

print('Ratings:', ratings.shape)
print('Movies:', movies.shape)

Ratings: (100836, 4)
Movies: (9742, 3)


## 3. Inspect the data

In [3]:
display(ratings.head())
display(movies.head())
print('Users:', ratings['userId'].nunique())
print('Movies with ratings:', ratings['movieId'].nunique())
print('Average rating:', round(ratings['rating'].mean(), 3))

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Users: 610
Movies with ratings: 9724
Average rating: 3.502


## 4. Prepare Surprise data

Surprise expects user ID, item ID and rating columns. Movie IDs are used directly; no large user-item matrix is materialized.

In [4]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
print('Training ratings:', trainset.n_ratings)
print('Test ratings:', len(testset))

Training ratings: 80668
Test ratings: 20168


## 5. Train the SVD model

The model uses a modest number of latent factors so that it remains practical on a CPU-only computer.

In [5]:
model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42,
)

model.fit(trainset)
print('SVD training completed.')

SVD training completed.


## 6. Evaluate the recommender

In [6]:
predictions = model.test(testset)
rmse(predictions, verbose=True)
mae(predictions, verbose=True)

RMSE: 0.8775
MAE:  0.6742


np.float64(0.6741748366526163)

## 7. Generate recommendations for a user

In [7]:
def recommend_movies(model, ratings, movies, user_id, n=10):
    rated = set(ratings.loc[ratings['userId'] == user_id, 'movieId'])
    candidates = movies.loc[~movies['movieId'].isin(rated), ['movieId', 'title']].copy()
    candidates['estimated_rating'] = candidates['movieId'].apply(
        lambda movie_id: model.predict(user_id, int(movie_id)).est
    )
    return candidates.sort_values('estimated_rating', ascending=False).head(n)

user_id = int(ratings['userId'].min())
recommendations = recommend_movies(model, ratings, movies, user_id)
display(recommendations)

,movieId,title,estimated_rating
906,1204,Lawrence of Arabia (1962),5.000000
878,1172,Cinema Paradiso (Nuovo cinema Paradiso) (1989),5.000000
1616,2160,Rosemary's Baby (1968),5.000000
1730,2324,Life Is Beautiful (La Vita è bella) (1997),5.000000
277,318,"Shawshank Redemption, The (1994)",5.000000
596,741,Ghost in the Shell (Kôkaku kidôtai) (1995),5.000000
2462,3275,"Boondock Saints, The (2000)",5.000000
6016,38061,Kiss Kiss Bang Bang (2005),5.000000
3622,4973,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",4.987584
6315,48516,"Departed, The (2006)",4.981134


## 8. Save the model and small metadata files

The Flask application loads these files directly. The original dataset does not need to be stored in the repository.

In [8]:
joblib.dump(model, MODEL_DIR / 'svd_recommender.pkl')
movies[['movieId', 'title']].to_csv(MODEL_DIR / 'movies.csv', index=False)
pd.DataFrame({'userId': sorted(ratings['userId'].unique())}).to_csv(
    MODEL_DIR / 'user_ids.csv', index=False
)

print('Saved model and metadata to:', MODEL_DIR)

Saved model and metadata to: C:\Users\user\Desktop\Matrix-Factorization-Recommender\models


## 9. Interview discussion points

- Explain collaborative filtering and the difference between user-based and item-based methods.
- Explain matrix factorization and latent user/item factors.
- Explain why SVD can predict ratings for unseen user-item pairs.
- Discuss RMSE and MAE and why offline metrics do not fully measure recommendation quality.
- Explain cold-start limitations for new users and new movies.
- Explain why a small MovieLens dataset is appropriate for a CPU-friendly portfolio project.
- Demonstrate the Flask application as the deployment layer.